# Bài 6.6 · Large language model trong một pipeline

**Câu hỏi của buổi này:** một LLM chỉ viết tiếp văn bản. Cần những gì quanh nó (template, quy định đầu ra, ví dụ, ngữ cảnh, bộ chấm) để nó làm đúng một nhiệm vụ, và đo bằng cách nào?

Model là họ **Qwen2.5** cỡ nhỏ chạy được trên một GPU T4: bản base 0.5B, bản instruct 0.5B và 1.5B. Dữ liệu: 40 thông báo tiếng Việt sinh từ mẫu (có đáp án), **SST-2** (câu nhận xét phim tiếng Anh, nhãn tích cực/tiêu cực) và **SQuAD** (câu hỏi về đoạn Wikipedia, có đáp án).

| Mục | Sẽ thấy |
|---|---|
| 1. Base và instruct | cùng một yêu cầu, hai kiểu trả lời; chat template thật sự trông thế nào |
| 2. Trích JSON, chấm ba mức | schema hợp lệ, đúng từng trường, đúng cả câu; so với baseline regex |
| 3. Few-shot | accuracy SST-2 theo số ví dụ, và số token phải trả mỗi lần gọi |
| 4. Ngữ cảnh và RAG | không ngữ cảnh, đoạn truy hồi, đoạn đúng; lỗi nằm ở retriever hay ở model |
| 5. Tốc độ trên T4 | token mỗi giây theo cỡ model và cỡ batch, dự toán thời gian |
| 6. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 6 phút. Bài giảng: [ai-canuc80k · bài 6.6](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/6.6).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import json, re, time, html
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
transformers.logging.set_verbosity_error()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if device == 'cuda' else torch.float32
BASE, SMALL, LARGE = 'Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct'
N_SST, N_QA, SHOTS = 300, 200, [0, 1, 4, 16]
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')], columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

def load(name):
    with quiet():
        tok = AutoTokenizer.from_pretrained(name, padding_side='left')
        model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE).to(device).eval()
    model.generation_config.temperature = model.generation_config.top_p = model.generation_config.top_k = None
    return tok, model

def chat(tok, user, system=None, shots=()):
    msgs = ([{'role': 'system', 'content': system}] if system else [])
    for u, a in shots: msgs += [{'role': 'user', 'content': u}, {'role': 'assistant', 'content': a}]
    return tok.apply_chat_template(msgs + [{'role': 'user', 'content': user}], tokenize=False, add_generation_prompt=True)

def generate(tok, model, prompts, max_new=48, bs=16):
    out = []
    for i in range(0, len(prompts), bs):
        b = tok(prompts[i:i + bs], return_tensors='pt', padding=True).to(device)
        with torch.no_grad(): o = model.generate(**b, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        out += tok.batch_decode(o[:, b['input_ids'].shape[1]:], skip_special_tokens=True)   # chỉ lấy phần sinh thêm
    return [x.strip() for x in out]

t = time.time()
tok_b, base = load(BASE); tok_s, small = load(SMALL); tok_l, large = load(LARGE)
sizes = pd.DataFrame([(n, sum(p.numel() for p in m.parameters()) / 1e9) for n, m in [(BASE, base), (SMALL, small), (LARGE, large)]], columns=['model', 'tỉ tham số'])
table(sizes, f'Ba checkpoint (tải và nạp {time.time() - t:.0f} giây)', fmt={'tỉ tham số': '{:.2f}'})

## 1. Base model viết tiếp, instruct model làm theo

Đưa cùng một yêu cầu cho bản base (chỉ pretrain) và bản instruct (đã post-training). Bản base nhận chuỗi thô; bản instruct nhận chuỗi qua **chat template** của chính nó.

In [ ]:
REQUEST = 'Trích JSON ngày và số học sinh: Buổi sinh hoạt ngày 12/10 có 18 học sinh.'
out_base = generate(tok_b, base, [REQUEST], max_new=40)[0]
templ = chat(tok_s, REQUEST)
out_inst = generate(tok_s, small, [templ], max_new=40)[0]
table(pd.DataFrame([('base 0.5B, chuỗi thô', out_base), ('instruct 0.5B, qua chat template', out_inst)], columns=['model', 'phần sinh thêm']), f'Yêu cầu: "{REQUEST}"')
display(HTML('<div style="font-weight:600;margin:10px 0 2px">Chuỗi thật sự đưa vào model instruct (special token hiện nguyên)</div>'
             f'<pre style="font-size:12.5px;line-height:1.35;white-space:pre-wrap">{html.escape(templ)}</pre>'))
b = tok_s(templ, return_tensors='pt').to(device)
with torch.no_grad(): o = small.generate(**b, max_new_tokens=40, do_sample=False, pad_token_id=tok_s.pad_token_id)
full = tok_s.decode(o[0], skip_special_tokens=True)
check('decode cả output thì prompt lẫn vào câu trả lời', full.startswith('system') or REQUEST in full)
check('cắt từ độ dài input thì chỉ còn phần sinh thêm', tok_s.decode(o[0, b['input_ids'].shape[1]:], skip_special_tokens=True).strip() == out_inst)
note(f'Bản base không coi đây là yêu cầu mà viết tiếp như một đoạn văn bất kỳ ({len(out_base)} ký tự, {"có" if "{" in out_base else "không có"} dấu ngoặc JSON). '
     f'Bản instruct nhận cùng nội dung bọc trong các token <code>&lt;|im_start|&gt;</code>, <code>&lt;|im_end|&gt;</code> và vai <code>system</code>/<code>user</code>/<code>assistant</code> mà nó đã học lúc post-training. '
     f'Nếu decode cả <code>out[0]</code> thì chuỗi trả về bắt đầu bằng prompt ({len(full)} ký tự thay vì {len(out_inst)}), parser JSON hỏng ngay ký tự đầu.')

## 2. Quy định đầu ra, rồi chấm ba mức

Nhiệm vụ: từ một thông báo, trả **một dòng JSON** `{"date": "d/m" hoặc null, "students": số nguyên hoặc null}`, lấy thông tin **thực tế** (nếu kế hoạch bị đổi). Notebook sinh 40 thông báo từ 5 mẫu, mỗi mẫu 8 câu: dễ, gọi là "em", chưa có ngày, kế hoạch bị đổi, số viết bằng chữ.

Bộ chấm có ba mức: **schema hợp lệ** (parse được, đúng hai khóa, đúng kiểu), **đúng từng trường**, **đúng cả câu**. Mốc so sánh là baseline regex của bài học.

In [ ]:
rng = np.random.default_rng(0)
ACTS = ['sinh hoạt', 'ngoại khóa', 'luyện đề', 'đọc sách', 'thi thử', 'dã ngoại', 'câu lạc bộ AI', 'văn nghệ']
WORDS = {12: 'mười hai', 15: 'mười lăm', 20: 'hai mươi', 24: 'hai mươi bốn', 30: 'ba mươi', 9: 'chín', 18: 'mười tám', 27: 'hai mươi bảy'}
notices = []
for i, act in enumerate(ACTS):
    d, m, n = int(rng.integers(1, 29)), int(rng.integers(1, 13)), int(rng.integers(8, 40))
    d2, n2 = min(d + int(rng.integers(1, 4)), 28), n - int(rng.integers(1, 5)); w = list(WORDS)[i]
    notices += [('dễ', f'Buổi {act} ngày {d}/{m} có {n} học sinh.', {'date': f'{d}/{m}', 'students': n}),
                ('gọi là "em"', f'Ngày {d}/{m}, có {n} em tham gia {act}.', {'date': f'{d}/{m}', 'students': n}),
                ('chưa có ngày', f'Buổi {act} chưa thông báo ngày, có {n} học sinh đăng ký.', {'date': None, 'students': n}),
                ('kế hoạch đổi', f'Kế hoạch {act} ngày {d}/{m} có {n} học sinh; thực tế chuyển sang ngày {d2}/{m} với {n2} học sinh.', {'date': f'{d2}/{m}', 'students': n2}),
                ('số bằng chữ', f'Buổi {act} ngày {d}/{m}: có tất cả {WORDS[w]} học sinh.', {'date': f'{d}/{m}', 'students': w})]
dev = pd.DataFrame(notices, columns=['loại', 'thông báo', 'gold'])

def validate(raw):
    try: a = json.loads(raw)
    except Exception: return None
    if not isinstance(a, dict) or set(a) != {'date', 'students'}: return None
    if not (a['date'] is None or (isinstance(a['date'], str) and re.fullmatch(r'\d{1,2}/\d{1,2}', a['date']))): return None
    if not (a['students'] is None or (isinstance(a['students'], int) and not isinstance(a['students'], bool))): return None
    return a

def regex_baseline(s):
    dm = re.search(r'ngày\s+(\d{1,2}/\d{1,2})', s, flags=re.I); cm = re.search(r'(\d+)\s+học sinh', s, flags=re.I)
    return json.dumps({'date': dm.group(1) if dm else None, 'students': int(cm.group(1)) if cm else None}, ensure_ascii=False)

SYSTEM = ('Trích ngày diễn ra thực tế và số học sinh tham gia từ thông báo. Nếu kế hoạch bị đổi, lấy thông tin thực tế. '
          'Chỉ trả về đúng một dòng JSON dạng {"date": "d/m" hoặc null, "students": số nguyên hoặc null}, không giải thích, không dùng ```.')
FEW = [('Buổi họp lớp ngày 3/9 có 25 học sinh.', '{"date": "3/9", "students": 25}'),
       ('Ngày 7/11, có 14 em tham gia trực nhật.', '{"date": "7/11", "students": 14}'),
       ('Kế hoạch tham quan ngày 2/4 có 30 học sinh; thực tế chuyển sang ngày 5/4 với 28 học sinh.', '{"date": "5/4", "students": 28}')]

def score(outputs):
    rows = []
    for raw, gold in zip(outputs, dev['gold']):
        a = validate(raw)
        rows.append({'schema': a is not None, 'date': a is not None and a['date'] == gold['date'],
                     'students': a is not None and a['students'] == gold['students'], 'exact': a == gold})
    return pd.DataFrame(rows)

t = time.time()
runs = {'baseline regex': [regex_baseline(s) for s in dev['thông báo']],
        'instruct 0.5B, chỉ dẫn': generate(tok_s, small, [chat(tok_s, s, SYSTEM) for s in dev['thông báo']]),
        'instruct 1.5B, chỉ dẫn': generate(tok_l, large, [chat(tok_l, s, SYSTEM) for s in dev['thông báo']]),
        'instruct 1.5B, chỉ dẫn + 3 ví dụ': generate(tok_l, large, [chat(tok_l, s, SYSTEM, FEW) for s in dev['thông báo']])}
t_ext = time.time() - t
sc = {k: score(v) for k, v in runs.items()}
res = pd.DataFrame([(k, v['schema'].mean(), v['date'].mean(), v['students'].mean(), v['exact'].mean()) for k, v in sc.items()],
                   columns=['cách làm', 'schema hợp lệ', 'đúng ngày', 'đúng số học sinh', 'đúng cả câu'])
table(res, f'{len(dev)} thông báo, chấm ba mức ({t_ext:.0f} giây cho ba lượt LLM)', fmt={c: '{:.0%}' for c in res.columns[1:]}, best='đúng cả câu')
by_type = pd.DataFrame({k: v['exact'].groupby(dev['loại']).mean() for k, v in sc.items()}).loc[['dễ', 'gọi là "em"', 'chưa có ngày', 'kế hoạch đổi', 'số bằng chữ']]
table(by_type.reset_index(names='loại thông báo'), 'Đúng cả câu theo loại (8 câu mỗi loại)', fmt={c: '{:.0%}' for c in by_type.columns})
best = res.loc[res['đúng cả câu'].idxmax(), 'cách làm']
k_bad = int(np.flatnonzero(~sc['instruct 0.5B, chỉ dẫn']['schema'].values)[0]) if (~sc['instruct 0.5B, chỉ dẫn']['schema']).any() else 0
ex = pd.DataFrame([(dev['thông báo'][k_bad], runs['instruct 0.5B, chỉ dẫn'][k_bad], runs[best][k_bad])], columns=['thông báo', 'instruct 0.5B', best])
table(ex, 'Một câu so hai output')
check('baseline regex luôn cho JSON hợp lệ', res.iloc[0, 1] == 1)
check('mọi output được chấm', all(len(v) == len(dev) for v in sc.values()))
def padded(run):     # sai ngày chỉ vì viết thêm số 0, như "24/08" thay cho "24/8"
    n = 0
    for raw, gold in zip(runs[run], dev['gold']):
        a = validate(raw)
        if a and a['date'] and gold['date'] and a['date'] != gold['date'] and [int(x) for x in a['date'].split('/')] == [int(x) for x in gold['date'].split('/')]: n += 1
    return n
pad_l, pad_f = padded('instruct 1.5B, chỉ dẫn'), padded('instruct 1.5B, chỉ dẫn + 3 ví dụ')
note(f'Regex luôn đúng schema nhưng chỉ đúng cả câu {res.iloc[0, 4]:.0%}: nó vấp ở cách gọi "em", ở kế hoạch đổi và ở số bằng chữ. '
     f'Instruct 0.5B: schema hợp lệ {res.iloc[1, 1]:.0%}, đúng cả câu {res.iloc[1, 4]:.0%}; 1.5B: {res.iloc[2, 1]:.0%} và {res.iloc[2, 4]:.0%}; thêm 3 ví dụ: {res.iloc[3, 1]:.0%} và {res.iloc[3, 4]:.0%}. '
     + (f'Ở bản 1.5B chỉ có chỉ dẫn, {pad_l}/{len(dev)} câu sai ngày chỉ vì viết thêm số 0 (như "24/08"); với 3 ví dụ cùng định dạng "d/m" còn {pad_f}. Quy định đầu ra phải ghi rõ định dạng, hoặc bộ chấm chuẩn hóa ngày nếu scorer của đề cho phép. ' if pad_l else '')
     + 'Tách ba mức mới biết nên sửa ở đâu: schema hỏng thì sửa quy định đầu ra hoặc thêm ví dụ, schema đúng mà sai trường thì là lỗi hiểu nội dung.')

## 3. Few-shot: ví dụ có giá, phải đo

SST-2: câu nhận xét phim tiếng Anh, đoán tích cực hay tiêu cực. Không sinh chữ, chỉ so xác suất model gán cho từ `positive` và `negative` ở token đầu tiên của câu trả lời. Đổi số ví dụ trong prompt: 0, 1, 4, 16 (ví dụ lấy từ train, cân bằng hai nhãn).

In [ ]:
with quiet():
    sst_val = pd.read_parquet('https://huggingface.co/api/datasets/stanfordnlp/sst2/parquet/default/validation/0.parquet').sample(N_SST, random_state=0)
    sst_tr = pd.read_parquet('https://huggingface.co/api/datasets/stanfordnlp/sst2/parquet/default/train/0.parquet')
pool = pd.concat([sst_tr[sst_tr.label == l].sample(8, random_state=1) for l in (0, 1)]).sample(frac=1, random_state=2)
ASK = 'Is the sentiment of this movie review positive or negative? Answer with one word.\n\nReview: {}'
LAB = ['negative', 'positive']

def sst_eval(tok, model, k, bs=16):
    shots = [(ASK.format(r.sentence.strip()), LAB[r.label]) for r in pool.head(k).itertuples()]
    ids = [tok.encode(w, add_special_tokens=False)[0] for w in LAB]
    prompts = [chat(tok, ASK.format(s.strip()), shots=shots) for s in sst_val['sentence']]
    preds, n_tok = [], []
    for i in range(0, len(prompts), bs):
        b = tok(prompts[i:i + bs], return_tensors='pt', padding=True).to(device)
        with torch.no_grad(): logit = model(**b).logits[:, -1, ids]
        preds += logit.argmax(1).tolist(); n_tok += b['attention_mask'].sum(1).tolist()
    return (np.array(preds) == sst_val['label'].values).mean(), np.mean(n_tok)

t = time.time()
fs = pd.DataFrame([(k, *sst_eval(tok_s, small, k), *sst_eval(tok_l, large, k)) for k in SHOTS], columns=['số ví dụ', 'acc 0.5B', 'token 0.5B', 'acc 1.5B', 'token 1.5B'])
fig, ax = plt.subplots(figsize=(9, 3.8)); ax2 = ax.twinx()
ax.plot(fs['số ví dụ'], fs['acc 0.5B'], 'o-', color=C['blue'], lw=2, label='instruct 0.5B')
ax.plot(fs['số ví dụ'], fs['acc 1.5B'], 'o-', color=C['clay'], lw=2, label='instruct 1.5B')
ax2.bar(fs['số ví dụ'], fs['token 1.5B'], width=.6, color=C['gray'], alpha=.3)
ax.set_xscale('symlog'); ax.set_xticks(SHOTS, SHOTS); ax.set(xlabel='số ví dụ trong prompt', ylabel='accuracy', title=f'SST-2, {N_SST} câu validation ({time.time() - t:.0f} giây)')
ax2.set_ylabel('token mỗi lần gọi (cột xám)'); ax2.grid(False); ax.legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.show()
table(fs, 'Accuracy và độ dài prompt', fmt={'acc 0.5B': '{:.3f}', 'acc 1.5B': '{:.3f}', 'token 0.5B': '{:.0f}', 'token 1.5B': '{:.0f}'})
check('prompt dài ra theo số ví dụ', fs['token 1.5B'].is_monotonic_increasing)
gain = fs['acc 1.5B'].max() - fs['acc 1.5B'].iloc[0]
note(f'Không ví dụ nào, 1.5B đã đúng {fs["acc 1.5B"].iloc[0]:.1%}; tốt nhất {fs["acc 1.5B"].max():.1%} với {int(fs.loc[fs["acc 1.5B"].idxmax(), "số ví dụ"])} ví dụ ({gain:+.1%}). '
     f'Mỗi lần gọi với {SHOTS[-1]} ví dụ tốn {fs["token 1.5B"].iloc[-1]:.0f} token thay vì {fs["token 1.5B"].iloc[0]:.0f}: ví dụ nằm trong prompt nên phải trả ở mọi lần gọi. '
     + ('Model 0.5B được lợi từ ví dụ nhiều hơn model 1.5B. ' if (fs['acc 0.5B'].max() - fs['acc 0.5B'].iloc[0]) > gain else 'Model 1.5B được lợi từ ví dụ ít nhất bằng model 0.5B. '))

## 4. Ngữ cảnh có hạn: RAG và hai tầng lỗi

SQuAD: câu hỏi về một đoạn Wikipedia, đáp án là một cụm từ trong đoạn. Kho tìm kiếm là **mọi đoạn** của tập validation. Ba cách hỏi model 1.5B:

- **không ngữ cảnh**: chỉ câu hỏi, model trả lời từ trọng số;
- **RAG**: sentence embedding (bài 6.2) lấy 3 đoạn gần câu hỏi nhất, đặt vào prompt;
- **đoạn đúng**: đưa đúng đoạn chứa đáp án (trần của tầng đọc).

Câu trả lời được tính đúng nếu chứa một trong các đáp án gold (sau khi viết thường, bỏ dấu câu).

In [ ]:
with quiet():
    sq = pd.read_parquet('https://huggingface.co/api/datasets/rajpurkar/squad/parquet/plain_text/validation/0.parquet')
    emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
corpus = sq['context'].drop_duplicates().tolist(); cid = {c: i for i, c in enumerate(corpus)}
qa = sq.sample(N_QA, random_state=0).reset_index(drop=True)
t = time.time()
E = emb.encode(corpus, batch_size=128, normalize_embeddings=True, convert_to_tensor=True)
Q = emb.encode(qa['question'].tolist(), batch_size=128, normalize_embeddings=True, convert_to_tensor=True)
top = (Q @ E.T).topk(3).indices.cpu().numpy()
gold_id = qa['context'].map(cid).values
hit1, hit3 = (top[:, 0] == gold_id), (top == gold_id[:, None]).any(1)
t_ret = time.time() - t

QA_SYS = 'Answer the question with a short phrase only. If context is given, answer only from the context.'
def ask(question, passages=()):
    ctx = '\n\n'.join(f'[{j + 1}] {p}' for j, p in enumerate(passages))
    return chat(tok_l, (f'Context:\n{ctx}\n\n' if passages else '') + f'Question: {question}', QA_SYS)
norm_ = lambda s: re.sub(r'[^a-z0-9 ]', ' ', s.lower()).split()
def correct(out, golds): o = ' '.join(norm_(out)); return any(' '.join(norm_(g)) in o for g in golds)
t = time.time()
cond = {'không ngữ cảnh': [ask(q) for q in qa['question']],
        'RAG: 3 đoạn truy hồi': [ask(q, [corpus[j] for j in top[i]]) for i, q in enumerate(qa['question'])],
        'đoạn đúng': [ask(q, [c]) for q, c in zip(qa['question'], qa['context'])]}
ans = {k: generate(tok_l, large, v, max_new=24, bs=8) for k, v in cond.items()}
t_gen = time.time() - t
ok = {k: np.array([correct(a, g['text']) for a, g in zip(v, qa['answers'])]) for k, v in ans.items()}
rag = pd.DataFrame([(k, v.mean()) for k, v in ok.items()], columns=['cách hỏi', 'đúng'])
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={'width_ratios': [1.3, 1]})
axes[0].barh(rag['cách hỏi'][::-1], rag['đúng'][::-1], color=[C['green'], C['blue'], C['gray']])
for i, v in enumerate(rag['đúng'][::-1]): axes[0].text(v + .01, i, f'{v:.0%}', va='center')
axes[0].set_xlim(0, 1.1); axes[0].set(xlabel='tỉ lệ câu trả lời chứa đáp án gold', title=f'{N_QA} câu hỏi SQuAD, kho {len(corpus)} đoạn')
r = ok['RAG: 3 đoạn truy hồi']
split_ = pd.DataFrame({'trường hợp': ['đoạn đúng có trong top-3', 'đoạn đúng không có trong top-3'], 'số câu': [int(hit3.sum()), int((~hit3).sum())],
                       'RAG đúng': [r[hit3].mean(), r[~hit3].mean() if (~hit3).any() else np.nan]})
axes[1].bar([f'{t_}\n({n} câu)' for t_, n in zip(split_['trường hợp'], split_['số câu'])], split_['RAG đúng'].fillna(0), color=[C['blue'], C['sand']]); bar_labels(axes[1], '{:.0%}')
axes[1].set(ylim=(0, 1.1), title='RAG tách theo tầng truy hồi')
plt.tight_layout(); plt.show()
table(pd.DataFrame([('recall@1', hit1.mean()), ('recall@3', hit3.mean())], columns=['retriever MiniLM', 'giá trị']), f'Tầng truy hồi ({t_ret:.0f} giây cho {len(corpus)} đoạn)', fmt={'giá trị': '{:.1%}'})
k_wrong = next((i for i in range(N_QA) if not ok['không ngữ cảnh'][i] and ok['đoạn đúng'][i]), 0)
table(pd.DataFrame([(qa['question'][k_wrong], ' / '.join(dict.fromkeys(qa['answers'][k_wrong]['text'])), ans['không ngữ cảnh'][k_wrong], ans['đoạn đúng'][k_wrong])],
                   columns=['câu hỏi', 'gold', 'không ngữ cảnh', 'có đoạn đúng']), 'Một câu model trả lời trôi chảy nhưng sai khi không có nguồn')
check('có đoạn đúng thì tốt hơn không ngữ cảnh', ok['đoạn đúng'].mean() > ok['không ngữ cảnh'].mean())
note(f'Không ngữ cảnh, model 1.5B chỉ đúng {ok["không ngữ cảnh"].mean():.0%}; có đoạn đúng thì {ok["đoạn đúng"].mean():.0%}; RAG {r.mean():.0%}. '
     f'Retriever đưa được đoạn đúng vào top-3 ở {hit3.mean():.0%} câu; ở các câu ấy RAG đúng {split_.iloc[0, 2]:.0%}, ở các câu thiếu đoạn đúng chỉ {split_.iloc[1, 2]:.0%}. '
     + f'Tách hai tầng: {1 - hit3.mean():.0%} câu mất ngay ở retriever (không có đoạn đúng trong prompt), còn khi đã có đúng đoạn thì model vẫn sai {1 - ok["đoạn đúng"].mean():.0%}. '
     + ('Retriever là nút thắt lớn hơn: sửa nó trước khi đổi LLM.' if 1 - hit3.mean() > 1 - ok['đoạn đúng'].mean() else 'Tầng đọc là nút thắt lớn hơn: retriever tốt hơn cũng không vượt được trần của model.'))

## 5. Tốc độ trên T4: đo rồi mới dự toán

Sinh 64 token mới cho prompt khoảng 100 token, batch 1 và batch 16, với hai cỡ model instruct.

In [ ]:
def speed(tok, model, bs, new=64):
    p = [chat(tok, 'Viết một đoạn ngắn về lợi ích của việc đọc sách đối với học sinh trung học.')] * bs
    b = tok(p, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        model.generate(**b, max_new_tokens=8, do_sample=False, pad_token_id=tok.pad_token_id)
        if device == 'cuda': torch.cuda.synchronize()
        t = time.time(); o = model.generate(**b, max_new_tokens=new, min_new_tokens=new, do_sample=False, pad_token_id=tok.pad_token_id)
        if device == 'cuda': torch.cuda.synchronize()
    dt = time.time() - t
    return b['input_ids'].shape[1], dt, bs * new / dt
sp = pd.DataFrame([(n, bs, *speed(tk, m, bs)) for n, tk, m in [('instruct 0.5B', tok_s, small), ('instruct 1.5B', tok_l, large)] for bs in (1, 16)],
                  columns=['model', 'batch', 'token prompt', 'giây', 'token mới / giây'])
sp['1000 thông báo × 64 token (phút)'] = 1000 * 64 / sp['token mới / giây'] / 60
table(sp, 'Tốc độ sinh ' + ('trên ' + torch.cuda.get_device_name(0) if device == 'cuda' else 'trên CPU'), fmt={'giây': '{:.2f}', 'token mới / giây': '{:.0f}', '1000 thông báo × 64 token (phút)': '{:.1f}'})
check('batch 16 cho nhiều token mỗi giây hơn batch 1', all(sp[(sp.model == n) & (sp.batch == 16)]['token mới / giây'].values > sp[(sp.model == n) & (sp.batch == 1)]['token mới / giây'].values for n in ['instruct 0.5B', 'instruct 1.5B']))
r1 = sp.set_index(['model', 'batch'])['token mới / giây']
note(f'Model 1.5B batch 1 sinh {r1[("instruct 1.5B", 1)]:.0f} token/giây, batch 16 lên {r1[("instruct 1.5B", 16)]:.0f}: '
     f'1 000 thông báo mất {sp.iloc[2, 5]:.1f} phút nếu gọi từng cái, {sp.iloc[3, 5]:.1f} phút nếu gom batch. '
     f'Sinh token là từng bước tuần tự, nên gom nhiều câu vào một batch là cách rẻ nhất để chạy kịp giờ thi.')

## 6. Tóm tắt

In [ ]:
summary = [('trích JSON, đúng cả câu: regex / 0.5B / 1.5B / 1.5B + 3 ví dụ', ' / '.join(f'{v:.0%}' for v in res['đúng cả câu'])),
           ('trích JSON, schema hợp lệ: 0.5B / 1.5B', f'{res.iloc[1, 1]:.0%} / {res.iloc[2, 1]:.0%}'),
           ('SST-2 1.5B: 0 ví dụ / tốt nhất', f'{fs["acc 1.5B"].iloc[0]:.1%} / {fs["acc 1.5B"].max():.1%}'),
           ('token mỗi lần gọi: 0 / 16 ví dụ', f'{fs["token 1.5B"].iloc[0]:.0f} / {fs["token 1.5B"].iloc[-1]:.0f}'),
           ('SQuAD: không ngữ cảnh / RAG / đoạn đúng', f'{ok["không ngữ cảnh"].mean():.0%} / {r.mean():.0%} / {ok["đoạn đúng"].mean():.0%}'),
           ('retriever recall@3', f'{hit3.mean():.0%}'),
           ('1.5B: token/giây batch 1 / batch 16', f'{r1[("instruct 1.5B", 1)]:.0f} / {r1[("instruct 1.5B", 16)]:.0f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ở mục 2 bỏ cụm "không dùng ```" trong chỉ dẫn và xem schema hợp lệ tụt bao nhiêu; ở mục 4 đổi top-3 thành top-1 hoặc top-10; ở mục 3 đổi thứ tự 16 ví dụ để xem accuracy dao động.